In [3]:
import oracledb
import pandas as pd


In [25]:
connection = oracledb.connect(
    user="olist_dwh",
    password="dwh12345",
    dsn="localhost:1521/XEPDB1"
)

print("Connected to Oracle Database!")


Connected to Oracle Database!


**ROLLUP with GROUPING()**

In [5]:
query = """
SELECT
    CASE
        WHEN GROUPING(p.category_english) = 1
            THEN 'All Categories'
        ELSE p.category_english
    END AS category,

    CASE
        WHEN GROUPING(s.seller_state) = 1
            THEN 'All States'
        ELSE s.seller_state
    END AS seller_state,

    SUM(f.payment_value) AS total_payment

FROM FACT_ORDERS f

JOIN DIM_PRODUCT p
    ON f.product_key = p.product_key

JOIN DIM_SELLER s
    ON f.seller_key = s.seller_key

GROUP BY ROLLUP(
    p.category_english,
    s.seller_state
)

ORDER BY
    p.category_english,
    s.seller_state
"""

df = pd.read_sql(query, connection)

df

C:\Users\Ashan\AppData\Local\Temp\ipykernel_12512\1345597732.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,CATEGORY,SELLER_STATE,TOTAL_PAYMENT
0,agro_industry_and_commerce,CE,515.08
1,agro_industry_and_commerce,MG,2570.20
2,agro_industry_and_commerce,PR,41138.85
3,agro_industry_and_commerce,RJ,494.45
4,agro_industry_and_commerce,RS,40596.22
...,...,...,...
579,watches_gifts,RS,5195.27
580,watches_gifts,SC,42382.60
581,watches_gifts,SP,1107861.71
582,watches_gifts,All States,1429216.68


* Make the output easier to read

In [7]:
print(df.to_string(index=False))

                               CATEGORY SELLER_STATE  TOTAL_PAYMENT
             agro_industry_and_commerce           CE         515.08
             agro_industry_and_commerce           MG        2570.20
             agro_industry_and_commerce           PR       41138.85
             agro_industry_and_commerce           RJ         494.45
             agro_industry_and_commerce           RS       40596.22
             agro_industry_and_commerce           SC          47.62
             agro_industry_and_commerce           SP       33368.19
             agro_industry_and_commerce   All States      118730.61
                       air_conditioning           DF        4297.84
                       air_conditioning           ES        3500.07
                       air_conditioning           MG        5359.74
                       air_conditioning           MS         184.99
                       air_conditioning           PR        9508.89
                       air_conditioning         

**Year-over-Year Comparison using LAG()**

In [8]:
query = """
WITH quarterly_revenue AS (
    SELECT
        p.category_english AS category,
        t.year,
        t.quarter,
        SUM(f.payment_value) AS revenue
    FROM FACT_ORDERS f
    JOIN DIM_PRODUCT p
        ON f.product_key = p.product_key
    JOIN DIM_TIME t
        ON f.time_key = t.time_key
    WHERE t.year IN (2017, 2018)
    GROUP BY
        p.category_english,
        t.year,
        t.quarter
)

SELECT
    category,
    year,
    quarter,
    revenue,

    LAG(revenue) OVER (
        PARTITION BY category, quarter
        ORDER BY year
    ) AS prior_year_revenue,

    ROUND(
        (
            revenue -
            LAG(revenue) OVER (
                PARTITION BY category, quarter
                ORDER BY year
            )
        ) * 100.0
        /
        NULLIF(
            LAG(revenue) OVER (
                PARTITION BY category, quarter
                ORDER BY year
            ),
            0
        ),
        2
    ) AS yoy_pct_change

FROM quarterly_revenue

ORDER BY
    category,
    quarter,
    year
"""

df = pd.read_sql(query, connection)

df

C:\Users\Ashan\AppData\Local\Temp\ipykernel_12512\1389150501.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,CATEGORY,YEAR,QUARTER,REVENUE,PRIOR_YEAR_REVENUE,YOY_PCT_CHANGE
0,agro_industry_and_commerce,2017,Q1,610.97,NaN,NaN
1,agro_industry_and_commerce,2018,Q1,25390.24,610.97,4055.73
2,agro_industry_and_commerce,2017,Q2,3185.79,NaN,NaN
3,agro_industry_and_commerce,2018,Q2,12059.93,3185.79,278.55
4,agro_industry_and_commerce,2017,Q3,5064.88,NaN,NaN
...,...,...,...,...,...,...
464,watches_gifts,2017,Q2,103707.65,NaN,NaN
465,watches_gifts,2018,Q2,368971.06,103707.65,255.78
466,watches_gifts,2017,Q3,143000.27,NaN,NaN
467,watches_gifts,2018,Q3,195537.41,143000.27,36.74


In [13]:
df["REVENUE"] = df["REVENUE"].round(2)
df["PRIOR_YEAR_REVENUE"] = df["PRIOR_YEAR_REVENUE"].round(2)
df["YOY_PCT_CHANGE"] = df["YOY_PCT_CHANGE"].round(2)

df

,CATEGORY,YEAR,QUARTER,REVENUE,PRIOR_YEAR_REVENUE,YOY_PCT_CHANGE
0,agro_industry_and_commerce,2017,Q1,610.97,NaN,NaN
1,agro_industry_and_commerce,2018,Q1,25390.24,610.97,4055.73
2,agro_industry_and_commerce,2017,Q2,3185.79,NaN,NaN
3,agro_industry_and_commerce,2018,Q2,12059.93,3185.79,278.55
4,agro_industry_and_commerce,2017,Q3,5064.88,NaN,NaN
...,...,...,...,...,...,...
464,watches_gifts,2017,Q2,103707.65,NaN,NaN
465,watches_gifts,2018,Q2,368971.06,103707.65,255.78
466,watches_gifts,2017,Q3,143000.27,NaN,NaN
467,watches_gifts,2018,Q3,195537.41,143000.27,36.74


**Ranking within a Dimension**

In [15]:
query = """
WITH seller_revenue AS (
    SELECT
        p.category_english AS category,
        s.seller_id,
        s.seller_state,
        SUM(f.payment_value) AS total_revenue
    FROM FACT_ORDERS f

    JOIN DIM_PRODUCT p
        ON f.product_key = p.product_key

    JOIN DIM_SELLER s
        ON f.seller_key = s.seller_key

    JOIN DIM_TIME t
        ON f.time_key = t.time_key

    WHERE t.year = 2018

    GROUP BY
        p.category_english,
        s.seller_id,
        s.seller_state
),

ranked AS (
    SELECT
        category,
        seller_id,
        seller_state,
        total_revenue,

        RANK() OVER (
            PARTITION BY category
            ORDER BY total_revenue DESC
        ) AS seller_rank

    FROM seller_revenue
)

SELECT
    category,
    seller_id,
    seller_state,
    total_revenue,
    seller_rank

FROM ranked

WHERE seller_rank <= 5

ORDER BY
    category,
    seller_rank
"""

df = pd.read_sql(query, connection)

df

C:\Users\Ashan\AppData\Local\Temp\ipykernel_12512\3497720883.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,CATEGORY,SELLER_ID,SELLER_STATE,TOTAL_REVENUE,SELLER_RANK
0,agro_industry_and_commerce,e59aa562b9f8076dd550fcddf0e73491,PR,30323.68,1
1,agro_industry_and_commerce,6bd69102ab48df500790a8cecfc285c2,SP,2552.95,2
2,agro_industry_and_commerce,6481e96574816ead57975da2c0f6d80d,SP,2302.43,3
3,agro_industry_and_commerce,cfd7ddab722b902f7ac5b5f3ba6d723d,MG,2268.88,4
4,agro_industry_and_commerce,d17f467e4bf608a510c20d82f4ba3b6b,RS,2253.82,5
...,...,...,...,...,...
322,watches_gifts,4869f7a5dfa277a7dca6462dcf3b52b2,SP,135625.08,1
323,watches_gifts,7d13fca15225358621be4086e1eb0964,SP,102717.47,2
324,watches_gifts,fa1c13f2614d7b5c4749cbc52fecda94,SP,100910.70,3
325,watches_gifts,6560211a19b47992c3666cc44a7e94c0,SP,84537.73,4


* Check the result

In [19]:
print(f"Number of results: {len(df)}")
df.head(20)

Number of results: 327


,CATEGORY,SELLER_ID,SELLER_STATE,TOTAL_REVENUE,SELLER_RANK
0,agro_industry_and_commerce,e59aa562b9f8076dd550fcddf0e73491,PR,30323.68,1
1,agro_industry_and_commerce,6bd69102ab48df500790a8cecfc285c2,SP,2552.95,2
2,agro_industry_and_commerce,6481e96574816ead57975da2c0f6d80d,SP,2302.43,3
3,agro_industry_and_commerce,cfd7ddab722b902f7ac5b5f3ba6d723d,MG,2268.88,4
4,agro_industry_and_commerce,d17f467e4bf608a510c20d82f4ba3b6b,RS,2253.82,5
5,air_conditioning,fcdd820084f17e9982427971e4e9d47f,SP,17833.42,1
6,air_conditioning,6a51fc556dab5f766ced6fbc860bc613,SP,3677.62,2
7,air_conditioning,7a241947449cc45dbfda4f9d0798d9d0,MG,3542.41,3
8,air_conditioning,15aac934c58d886785ac1b17953ea898,ES,3500.07,4
9,air_conditioning,f3b80352b986ab4d1057a4b724be19d0,DF,2227.91,5


**SLICE**

In [32]:
import sys
sys.path.append(r"C:\Users\Ashan\Desktop\Projects\olist-dwh-project\etl")
from db_connection import get_engine
engine = get_engine()

query = """
SELECT t.year, t.quarter, p.category_english, SUM(f.payment_value) AS revenue
FROM FACT_ORDERS f
JOIN DIM_TIME t ON f.time_key = t.time_key
JOIN DIM_PRODUCT p ON f.product_key = p.product_key
WHERE t.year = 2018 AND t.quarter = 'Q1'
GROUP BY t.year, t.quarter, p.category_english
"""

result = pd.read_sql(query, engine)
print(result)

    year quarter           category_english    revenue
0   2018      Q1              watches_gifts  289840.57
1   2018      Q1            furniture_decor  292027.86
2   2018      Q1                  telephony   84126.47
3   2018      Q1              fashion_shoes    4303.71
4   2018      Q1             bed_bath_table  351036.09
..   ...     ...                        ...        ...
64  2018      Q1             home_comfort_2     184.33
65  2018      Q1                 la_cuisine     556.73
66  2018      Q1        diapers_and_hygiene    2568.10
67  2018      Q1     fashio_female_clothing     859.65
68  2018      Q1  fashion_childrens_clothes     216.87

[69 rows x 4 columns]


In [42]:
# Query — SLICE
# Filters on one dimension only: customer_state = 'SP'
# This slices the cube down to just São Paulo, then shows city-level detail within that slice

query_slice = """
SELECT 
    c.customer_state,
    c.customer_city,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.price + f.freight_value) AS total_revenue
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
WHERE c.customer_state = 'SP'
GROUP BY c.customer_state, c.customer_city
ORDER BY total_revenue DESC
"""

df_slice = pd.read_sql(query_slice, engine)
print("Query — Slice (customer_state = 'SP'), drilled down to city")
display(df_slice)

Query — Slice (customer_state = 'SP'), drilled down to city


,customer_state,customer_city,total_orders,total_revenue
0,SP,sao paulo,15402,2170227.12
1,SP,campinas,1429,212541.70
2,SP,guarulhos,1178,163575.82
3,SP,sao bernardo do campo,928,119024.85
4,SP,santos,706,111670.21
...,...,...,...,...
623,SP,boraceia,1,39.96
624,SP,nova independencia,1,34.86
625,SP,nova luzitania,1,34.75
626,SP,aparecida d'oeste,1,31.75


**DRILL-DOWN**

In [ ]:

# Step 1: Summary level (customer_state)
query_state = """
SELECT 
    c.customer_state,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.price + f.freight_value) AS total_revenue
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
GROUP BY c.customer_state
ORDER BY total_revenue DESC
"""

df_state = pd.read_sql(query_state, engine)
print("Query 1 — Summary by State")
display(df_state)


# Step 2: Drill-down to customer_city (within the same hierarchy)
query_city = """
SELECT 
    c.customer_state,
    c.customer_city,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.price + f.freight_value) AS total_revenue
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
GROUP BY c.customer_state, c.customer_city
ORDER BY c.customer_state, total_revenue DESC
"""

df_city = pd.read_sql(query_city, engine)
print("\nQuery 2 — Drill-Down to City")
display(df_city)

Query 1 — Summary by State


,customer_state,total_orders,total_revenue
0,SP,41375,5921678.12
1,RJ,12762,2129681.98
2,MG,11544,1856161.49
3,RS,5432,885826.76
4,PR,4998,800935.44
5,BA,3358,611506.67
6,SC,3612,610213.60
7,DF,2125,353229.44
8,GO,2007,347706.93
9,ES,2025,324801.91



Query 2 — Drill-Down to City


,customer_state,customer_city,total_orders,total_revenue
0,AC,rio branco,70,16917.55
1,AC,cruzeiro do sul,3,1127.06
2,AC,senador guiomard,2,547.09
3,AC,xapuri,2,445.89
4,AC,manoel urbano,1,248.71
...,...,...,...,...
4295,TO,buriti do tocantins,1,86.79
4296,TO,novo jardim,1,60.98
4297,TO,nova olinda,1,56.79
4298,TO,combinado,1,54.63


In [35]:
# Check the actual column names in DIM_PRODUCT
df_check = pd.read_sql("SELECT * FROM DIM_PRODUCT WHERE ROWNUM <= 1", engine)
print(df_check.columns.tolist())

['product_key', 'product_id', 'category_english', 'category_portuguese', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [43]:
# Query — DICE (3 dimensions)
# Filters on three different dimensions simultaneously:
# customer_state IN (...), category_english IN (...), and year = 2018
# This creates a smaller "sub-cube" of the data — more restrictive than a slice

query_dice3 = """
SELECT 
    c.customer_state,
    p.category_english AS product_category,
    t.year,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.price + f.freight_value) AS total_revenue
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
JOIN DIM_PRODUCT p ON f.product_key = p.product_key
JOIN DIM_TIME t ON f.time_key = t.time_key
WHERE c.customer_state IN ('SP', 'RJ', 'MG')
  AND p.category_english IN ('electronics', 'computers_accessories')
  AND t.year = 2018
GROUP BY c.customer_state, p.category_english, t.year
ORDER BY c.customer_state, total_revenue DESC
"""

df_dice3 = pd.read_sql(query_dice3, engine)
print("Query — Dice (3 dimensions: state, category, year)")
display(df_dice3)

Query — Dice (3 dimensions: state, category, year)


,customer_state,product_category,year,total_orders,total_revenue
0,MG,computers_accessories,2018,549,78752.84
1,MG,electronics,2018,163,18214.42
2,RJ,computers_accessories,2018,504,72886.69
3,RJ,electronics,2018,276,21568.68
4,SP,computers_accessories,2018,1701,241075.14
5,SP,electronics,2018,646,40631.20


**DICE**

In [36]:
# Query 3 — DICE
# Filters on two dimensions at once: customer_state (3 values) AND product_category (2 values)

query_dice = """
SELECT 
    c.customer_state,
    p.category_english AS product_category,
    COUNT(DISTINCT f.order_id) AS total_orders,
    SUM(f.price + f.freight_value) AS total_revenue
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
JOIN DIM_PRODUCT p ON f.product_key = p.product_key
WHERE c.customer_state IN ('SP', 'RJ', 'MG')
  AND p.category_english IN ('electronics', 'computers_accessories')
GROUP BY c.customer_state, p.category_english
ORDER BY c.customer_state, total_revenue DESC
"""

df_dice = pd.read_sql(query_dice, engine)
print("Query 3 — Dice (3 states × 2 categories)")
display(df_dice)

Query 3 — Dice (3 states × 2 categories)


,customer_state,product_category,total_orders,total_revenue
0,MG,computers_accessories,877,129424.86
1,MG,electronics,260,26305.67
2,RJ,computers_accessories,859,141591.83
3,RJ,electronics,400,34427.21
4,SP,computers_accessories,2679,396872.47
5,SP,electronics,938,64787.98


**PIVOT**

In [41]:
# Query 4 — PIVOT
# Uses CASE WHEN to turn customer_state values (SP, RJ, MG) into separate columns
# One row per category, with each state's revenue side-by-side

query_pivot = """
SELECT 
    p.category_english AS product_category,
    SUM(CASE WHEN c.customer_state = 'SP' THEN f.price + f.freight_value ELSE 0 END) AS revenue_SP,
    SUM(CASE WHEN c.customer_state = 'RJ' THEN f.price + f.freight_value ELSE 0 END) AS revenue_RJ,
    SUM(CASE WHEN c.customer_state = 'MG' THEN f.price + f.freight_value ELSE 0 END) AS revenue_MG
FROM FACT_ORDERS f
JOIN DIM_CUSTOMER c ON f.customer_key = c.customer_key
JOIN DIM_PRODUCT p ON f.product_key = p.product_key
WHERE c.customer_state IN ('SP', 'RJ', 'MG')
GROUP BY p.category_english
ORDER BY p.category_english
"""

df_pivot = pd.read_sql(query_pivot, engine)
print("Query 4 — Pivot (states as columns)")
display(df_pivot)

Query 4 — Pivot (states as columns)


,product_category,revenue_sp,revenue_rj,revenue_mg
0,agro_industry_and_commerce,27789.48,9385.32,12661.48
1,air_conditioning,25447.92,18082.58,4827.45
2,art,16480.32,3654.50,1699.96
3,arts_and_craftmanship,1090.77,0.00,627.35
4,audio,19116.08,10331.91,6234.00
...,...,...,...,...
66,tablets_printing_image,2752.47,1059.27,996.24
67,telephony,121368.16,45619.00,42960.55
68,toys,210953.86,84533.13,68634.00
69,unknown,77565.41,28223.74,22770.79


✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅

**Partitioning**

*Range partitioning on ORDER_DATE_SK, splitting FACT_ORDERS by year*

In [66]:
# --- C8.1 — Partitioning Strategy ---
# Range partitioning on ORDER_DATE_SK, splitting FACT_ORDERS by year
# order_id is VARCHAR2(50) to match the real FACT_ORDERS table (Olist order IDs are text, not numbers)

ddl_partition = """
CREATE TABLE FACT_ORDERS_PART (
    order_id           VARCHAR2(50),
    order_date_sk      NUMBER,
    customer_key       NUMBER,
    product_key        NUMBER,
    seller_key         NUMBER,
    price               NUMBER,
    freight_value       NUMBER
)
PARTITION BY RANGE (order_date_sk) (
    PARTITION p_2016 VALUES LESS THAN (20170101),
    PARTITION p_2017 VALUES LESS THAN (20180101),
    PARTITION p_2018 VALUES LESS THAN (20190101),
    PARTITION p_max  VALUES LESS THAN (MAXVALUE)
)
"""

with engine.connect() as conn:
    conn.execute(text(ddl_partition))
    conn.commit()
print("Partitioned table created.")

Partitioned table created.


**1. Load sample data**

*Copies data from the original FACT_ORDERS into the new partitioned table.*

*Purpose: Put data into the partitions so we can test them.*

In [67]:
# --- Load sample data from the real FACT_ORDERS + DIM_TIME ---

with engine.connect() as conn:
    conn.execute(text("""
        INSERT INTO FACT_ORDERS_PART (order_id, order_date_sk, customer_key, product_key, seller_key, price, freight_value)
        SELECT f.order_id, 
               t.year * 10000 + t.month * 100 + t.day_of_month,
               f.customer_key, f.product_key, f.seller_key, f.price, f.freight_value
        FROM FACT_ORDERS f
        JOIN DIM_TIME t ON f.time_key = t.time_key
    """))
    conn.commit()
print("Sample data loaded into FACT_ORDERS_PART.")

Sample data loaded into FACT_ORDERS_PART.


**2. Gather statistics**

*Tells Oracle information about the table, such as how many rows are in each partition.*

*Purpose: Help Oracle choose the best query plan.*

In [68]:
# --- Gather statistics so the optimizer has accurate row counts ---

with engine.connect() as conn:
    conn.execute(text("""
        BEGIN
            DBMS_STATS.GATHER_TABLE_STATS(ownname => USER, tabname => 'FACT_ORDERS_PART');
        END;
    """))
    conn.commit()
print("Statistics gathered.")

Statistics gathered.


**3. Check partitions**

*Shows the partitions and their row counts.*

*Purpose: Confirm that the data was correctly divided by year.*

In [69]:
# --- Check partitions and row counts ---

df_parts = pd.read_sql("""
    SELECT partition_name, high_value, num_rows
    FROM user_tab_partitions
    WHERE table_name = 'FACT_ORDERS_PART'
    ORDER BY partition_position
""", engine)
display(df_parts)

,partition_name,high_value,num_rows
0,P_2016,20170101,370
1,P_2017,20180101,50864
2,P_2018,20190101,61416
3,P_MAX,MAXVALUE,0


**4. Demonstrate partition pruning**

*Oracle skips the partitions that it doesn't need to search*

In [70]:
# --- Demonstrate partition pruning: filter ON the partition key ---

with engine.connect() as conn:
    conn.execute(text("""
        EXPLAIN PLAN FOR
        SELECT * FROM FACT_ORDERS_PART WHERE order_date_sk >= 20180101
    """))
    conn.commit()

    result = conn.execute(text("SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY())"))
    for row in result:
        print(row[0])

Plan hash value: 3745644384
 
-------------------------------------------------------------------------------------------------------------
| Id  | Operation                | Name             | Rows  | Bytes | Cost (%CPU)| Time     | Pstart| Pstop |
-------------------------------------------------------------------------------------------------------------
|   0 | SELECT STATEMENT         |                  |  4700 |   279K|   241   (1)| 00:00:01 |       |       |
|   1 |  PARTITION RANGE ITERATOR|                  |  4700 |   279K|   241   (1)| 00:00:01 |     3 |     4 |
|*  2 |   TABLE ACCESS FULL      | FACT_ORDERS_PART |  4700 |   279K|   241   (1)| 00:00:01 |     3 |     4 |
-------------------------------------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   2 - filter("ORDER_DATE_SK" IS NOT NULL)


**5. Control Comparison**

*WHERE customer_key = 12345*

*customer_key is not the partition key, so Oracle may need to check all partitions.*

*2016 ✅*

*2017 ✅*

*2018 ✅*

*MAX  ✅*

*Purpose: Compare with Step 4 and show why partition pruning works.*

In [71]:
# --- Control comparison: filter NOT on the partition key ---
# Should scan all partitions, unlike the query above

with engine.connect() as conn:
    conn.execute(text("""
        EXPLAIN PLAN FOR
        SELECT * FROM FACT_ORDERS_PART WHERE customer_key = 12345
    """))
    conn.commit()

    result = conn.execute(text("SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY())"))
    for row in result:
        print(row[0])

Plan hash value: 86557299
 
--------------------------------------------------------------------------------------------------------
| Id  | Operation           | Name             | Rows  | Bytes | Cost (%CPU)| Time     | Pstart| Pstop |
--------------------------------------------------------------------------------------------------------
|   0 | SELECT STATEMENT    |                  |     1 |    61 |   788   (1)| 00:00:01 |       |       |
|   1 |  PARTITION RANGE ALL|                  |     1 |    61 |   788   (1)| 00:00:01 |     1 |     4 |
|*  2 |   TABLE ACCESS FULL | FACT_ORDERS_PART |     1 |    61 |   788   (1)| 00:00:01 |     1 |     4 |
--------------------------------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   2 - filter("CUSTOMER_KEY"=12345)


✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅

**BITMAP INDEX**

*1. Drop the index first if it already exists*

In [75]:
from sqlalchemy import text

# --- Drop the index first if it already exists, so this can be re-run safely ---
with engine.connect() as conn:
    try:
        conn.execute(text("DROP INDEX idx_bmp_review_score"))
        conn.commit()
        print("Old index dropped.")
    except Exception:
        print("No existing index — continuing.")

No existing index — continuing.


*2. Create the Bitmap Index*

*This creates a Bitmap Index on:*

*FACT_ORDERS → review_score*

*Your review_score has only a few possible values:*

*1,*
*2,*
*3,*
*4,*
*5*

*This is called low cardinality.*

**Why Bitmap?**

*Bitmap indexes are very suitable for columns with few distinct values.*

In [ ]:
# --- Bitmap Index Selection ---
# review_score has very low cardinality (values 1–5), a good fit for a bitmap index

ddl_bitmap_review = """
CREATE BITMAP INDEX idx_bmp_review_score
ON FACT_ORDERS (review_score)
"""

with engine.connect() as conn:
    conn.execute(text(ddl_bitmap_review))
    conn.commit()
print("Bitmap index on review_score created.")

Bitmap index on review_score created.


**3. Confirm the index**

*This asks Oracle:*

*"What indexes exist on FACT_ORDERS?"*

In [77]:
# --- Confirm the index exists and is type BITMAP ---

df_idx = pd.read_sql("""
    SELECT index_name, index_type, table_name
    FROM user_indexes
    WHERE table_name = 'FACT_ORDERS'
""", engine)
display(df_idx)

,index_name,index_type,table_name
0,SYS_C008462,NORMAL,FACT_ORDERS
1,IDX_FACT_CUSTOMER,NORMAL,FACT_ORDERS
2,IDX_FACT_PRODUCT,NORMAL,FACT_ORDERS
3,IDX_FACT_SELLER,NORMAL,FACT_ORDERS
4,IDX_FACT_TIME,NORMAL,FACT_ORDERS
5,IDX_BMP_REVIEW_SCORE,BITMAP,FACT_ORDERS


**4. Gather statistics**

Number of rows

Data distribution

Column statistics

In [78]:
# --- Gather statistics so the optimizer considers the index ---

with engine.connect() as conn:
    conn.execute(text("""
        BEGIN
            DBMS_STATS.GATHER_TABLE_STATS(ownname => USER, tabname => 'FACT_ORDERS');
        END;
    """))
    conn.commit()
print("Statistics gathered.")

Statistics gathered.


**5. Demonstrate the index usage**

*You're asking Oracle:*

*"How would you execute this query?"*

*The query searches for:*

*review_score = 5*

*Since you created a Bitmap Index on review_score, Oracle may choose to use that index*

In [79]:
# --- Demonstrate usage: EXPLAIN PLAN for a query filtering on review_score ---

with engine.connect() as conn:
    conn.execute(text("""
        EXPLAIN PLAN FOR
        SELECT COUNT(*) FROM FACT_ORDERS WHERE review_score = 5
    """))
    conn.commit()

    result = conn.execute(text("SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY())"))
    for row in result:
        print(row[0])

Plan hash value: 1704263387
 
----------------------------------------------------------------------------------------------------
| Id  | Operation                   | Name                 | Rows  | Bytes | Cost (%CPU)| Time     |
----------------------------------------------------------------------------------------------------
|   0 | SELECT STATEMENT            |                      |     1 |     3 |     7   (0)| 00:00:01 |
|   1 |  SORT AGGREGATE             |                      |     1 |     3 |            |          |
|   2 |   BITMAP CONVERSION COUNT   |                      | 63081 |   184K|     7   (0)| 00:00:01 |
|*  3 |    BITMAP INDEX SINGLE VALUE| IDX_BMP_REVIEW_SCORE |       |       |            |          |
----------------------------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   3 - access("REVIEW_SCORE"=5)


**WITHOUT the bitmap index**

*Without: the Operation column should show TABLE ACCESS FULL*

*FACT_ORDERS*

   *↓*

*Search many/all rows*

   *↓*

*Find review_score = 5*

   *↓*

*Return result*

In [80]:
from sqlalchemy import text

# --- WITHOUT the bitmap index ---
# Hide the index from the optimizer temporarily
with engine.connect() as conn:
    conn.execute(text("ALTER INDEX idx_bmp_review_score INVISIBLE"))
    conn.commit()

    conn.execute(text("""
        EXPLAIN PLAN FOR
        SELECT * FROM FACT_ORDERS WHERE review_score = 5
    """))
    conn.commit()

    result = conn.execute(text("SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY())"))
    print("=== WITHOUT bitmap index ===")
    for row in result:
        print(row[0])

=== WITHOUT bitmap index ===
Plan hash value: 4180159637
 
---------------------------------------------------------------------------------
| Id  | Operation         | Name        | Rows  | Bytes | Cost (%CPU)| Time     |
---------------------------------------------------------------------------------
|   0 | SELECT STATEMENT  |             | 63081 |  4804K|   379   (2)| 00:00:01 |
|*  1 |  TABLE ACCESS FULL| FACT_ORDERS | 63081 |  4804K|   379   (2)| 00:00:01 |
---------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   1 - filter("REVIEW_SCORE"=5)


**WITH the bitmap index**

*With: it should show something like BITMAP CONVERSION TO ROWIDS*

*and BITMAP INDEX SINGLE VALUE, feeding into a TABLE ACCESS BY INDEX ROWID*

In [81]:
# --- WITH the bitmap index ---
# Make the index visible again
with engine.connect() as conn:
    conn.execute(text("ALTER INDEX idx_bmp_review_score VISIBLE"))
    conn.commit()

    conn.execute(text("""
        EXPLAIN PLAN FOR
        SELECT * FROM FACT_ORDERS WHERE review_score = 5
    """))
    conn.commit()

    result = conn.execute(text("SELECT * FROM TABLE(DBMS_XPLAN.DISPLAY())"))
    print("=== WITH bitmap index ===")
    for row in result:
        print(row[0])

=== WITH bitmap index ===
Plan hash value: 4180159637
 
---------------------------------------------------------------------------------
| Id  | Operation         | Name        | Rows  | Bytes | Cost (%CPU)| Time     |
---------------------------------------------------------------------------------
|   0 | SELECT STATEMENT  |             | 63081 |  4804K|   379   (2)| 00:00:01 |
|*  1 |  TABLE ACCESS FULL| FACT_ORDERS | 63081 |  4804K|   379   (2)| 00:00:01 |
---------------------------------------------------------------------------------
 
Predicate Information (identified by operation id):
---------------------------------------------------
 
   1 - filter("REVIEW_SCORE"=5)


✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅

**Materialized View**

*calculates and stores the result, so future queries can be much faster.*

*Before create materialized view*

*SQL Plus*

*ALTER SESSION SET CONTAINER = XEPDB1;*

*ALTER USER system IDENTIFIED BY dwh12345;*

*GRANT CREATE MATERIALIZED VIEW TO olist_dwh;*

*exit;*

**1. Drop old Materialized View**

In [84]:
from sqlalchemy import text

# --- Drop the MV first if it already exists, so this can be re-run safely ---
with engine.connect() as conn:
    try:
        conn.execute(text("DROP MATERIALIZED VIEW mv_daily_sales_by_category"))
        conn.commit()
        print("Old materialized view dropped.")
    except Exception:
        print("No existing materialized view — continuing.")

No existing materialized view — continuing.


**2. Create the Materialized View**

In [93]:
with engine.connect() as conn:
    conn.execute(text(ddl_mv))
    conn.commit()
print("Materialized view created.")

Materialized view created.


**3. Confirm the MV exists**

*This checks the status of your Materialized View.*

In [94]:
# --- Confirm the MV exists ---

df_mv = pd.read_sql("""
    SELECT mview_name, last_refresh_date, staleness, compile_state
    FROM user_mviews
    WHERE mview_name = 'MV_DAILY_SALES_BY_CATEGORY'
""", engine)
display(df_mv)

,mview_name,last_refresh_date,staleness,compile_state
0,MV_DAILY_SALES_BY_CATEGORY,2026-09-08 15:56:44,FRESH,VALID


**4. Query the Materialized View directly**

This simply shows the data stored inside the MV.

Your columns are:

    full_date

    category_english

    order_count

    total_sales_amount

    total_payment_amount

For example:

2018-01-01 | computers | 15 | 5000 | 5200

It means:

On that date, the computers category had 15 orders and the calculated sales/payment totals are already stored.

In [95]:
    # --- Query the MV directly, to prove it holds correct pre-aggregated data ---

df_mv_data = pd.read_sql("""
    SELECT * FROM mv_daily_sales_by_category
    ORDER BY full_date, category_english
    FETCH FIRST 10 ROWS ONLY
""", engine)
display(df_mv_data)

,full_date,category_english,order_count,total_sales_amount,total_payment_amount
0,2016-09-04,furniture_decor,1,136.23,272.46
1,2016-09-05,telephony,1,75.06,75.06
2,2016-09-15,health_beauty,1,143.46,NaN
3,2016-10-02,baby,1,109.34,109.34
4,2016-10-03,fashion_shoes,1,40.95,40.95
5,2016-10-03,furniture_decor,2,225.73,225.73
6,2016-10-03,sports_leisure,3,128.43,128.43
7,2016-10-03,toys,1,154.57,154.57
8,2016-10-03,watches_gifts,1,45.46,45.46
9,2016-10-04,air_conditioning,3,826.69,1165.15


✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅


In [20]:
connection.close()

print("Connection closed.")

Connection closed.
